# 07b — Breakout Stop-loss And Regime Validation

Notebook này gom phần nghiên cứu breakout vào một flow ngắn hơn:
- chuẩn hóa feature cần cho breakout research
- backtest non-overlap và quét fixed stop / ATR stop
- tạo setup eligibility và volatility rule để notebook trade card dùng lại


In [1]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path


def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")


project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [2]:
# Cell: Import Breakout Research Modules
import pandas as pd

from src.backtest.breakout_research import run_breakout_research, save_breakout_research_outputs
from src.data.loaders import read_parquet


In [3]:
# Cell: Load Feature Frame
candidate_paths = [
    "data/signals/setup_signals.parquet",
    "data/processed/stock_features.parquet",
    "data/processed/stock_features_full.parquet",
    "data/processed/clean_daily_ohlcv.parquet",
]

features = pd.DataFrame()
loaded_from = None

for candidate_path in candidate_paths:
    if not Path(candidate_path).exists():
        continue
    candidate_df = read_parquet(candidate_path)
    if candidate_df.empty:
        continue
    features = candidate_df.copy()
    loaded_from = candidate_path
    break

if features.empty:
    raise FileNotFoundError(
        "Khong tim thay data dau vao cho notebook 07b. "
        "Hay tao du lieu truoc bang scripts/generate_features.py va scripts/generate_signals.py"
    )

if "date" not in features.columns:
    raise KeyError(
        f"File {loaded_from} da duoc doc nhung khong co cot 'date'. "
        f"Current columns: {features.columns.tolist()}"
    )

features["date"] = pd.to_datetime(features["date"])
print("Loaded features from:", loaded_from)
features.shape


Loaded features from: data/signals/setup_signals.parquet


(186827, 48)

In [4]:
# Cell: Run Breakout Research
results = run_breakout_research(
    features,
    signal_col="breakout_20d",
    holding_period=20,
    cost=0.004,
    stop_losses=[None, -0.05, -0.07, -0.10, -0.12],
    atr_multiples=[1.5, 2.0, 2.5],
)

pd.DataFrame([results["baseline_summary"]])


,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate
0,2056,0.500973,0.050734,0.00021,2.950545,-0.765905,186.331632,82.022671,2.271709,0.0


In [5]:
# Cell: Review Fixed Stop Results
display(results["fixed_stop_summary"])
display(pd.DataFrame([results["best_stop_row"]]))
display(results["by_symbol"].head(10))
display(results["by_volatility_regime"])


,stop_loss,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate
0,-0.05,2056,0.363327,0.042533,-0.054000,2.950545,-0.230252,153.266788,65.818499,2.328628,0.550097
1,no_stop,2056,0.500973,0.050734,0.000210,2.950545,-0.765905,186.331632,82.022671,2.271709,0.000000
2,-0.12,2056,0.479086,0.048184,-0.006621,2.950545,-0.347634,179.590698,80.525307,2.230239,0.198444
3,-0.07,2056,0.420720,0.043814,-0.025902,2.950545,-0.347634,163.514555,73.432728,2.226726,0.407101
4,-0.1,2056,0.463521,0.046654,-0.009746,2.950545,-0.347634,175.686116,79.766361,2.202509,0.273346


,stop_loss,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate
0,-0.05,2056,0.363327,0.042533,-0.054,2.950545,-0.230252,153.266788,65.818499,2.328628,0.550097


,symbol,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate
0,SVT,1,1.0,0.030417,0.030417,0.030417,0.030417,0.030417,0.0,inf,0.0
1,BAF,4,1.0,0.261694,0.237878,0.394513,0.176504,1.046774,0.0,inf,0.0
2,QNP,1,1.0,0.018578,0.018578,0.018578,0.018578,0.018578,0.0,inf,0.0
3,AG1,1,1.0,0.037322,0.037322,0.037322,0.037322,0.037322,0.0,inf,0.0
4,NNC,3,1.0,0.191763,0.224677,0.324125,0.026488,0.575290,0.0,inf,0.0
5,LAF,1,1.0,0.092633,0.092633,0.092633,0.092633,0.092633,0.0,inf,0.0
6,TVT,1,1.0,0.014390,0.014390,0.014390,0.014390,0.014390,0.0,inf,0.0
7,APS,2,1.0,0.362350,0.362350,0.637791,0.086909,0.724700,0.0,inf,0.0
8,SAB,3,1.0,0.033790,0.034386,0.044458,0.022526,0.101370,0.0,inf,0.0
9,ATG,1,1.0,0.402250,0.402250,0.402250,0.402250,0.402250,0.0,inf,0.0


,volatility_regime,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate
0,Normal Vol,169,0.520710,0.102885,0.023095,2.046832,-0.135963,21.584070,4.196527,5.143318,0.372781
1,Low Vol,98,0.520408,0.086486,0.020126,1.901329,-0.078015,10.663616,2.188008,4.873663,0.367347
2,Panic Vol,39,0.538462,0.061039,0.012370,0.648533,-0.054000,3.352528,0.972000,3.449103,0.461538
3,Stress Vol,67,0.417910,0.066270,-0.043337,1.763189,-0.054000,6.367312,1.927249,3.303835,0.477612
4,High Vol,137,0.401460,0.054656,-0.054000,1.597785,-0.230252,12.098950,4.611088,2.623882,0.518248
5,Unknown,1546,0.326003,0.030580,-0.054000,2.950545,-0.078954,99.200312,51.923627,1.910504,0.589263


In [6]:
# Cell: Review ATR And Rules
display(results["atr_stop_summary"])
display(results["eligibility"])
display(results["volatility_rule"])


,atr_multiple,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate
0,1.5,1854,0.316613,0.027423,-0.035137,2.950545,-0.347634,113.206688,62.364981,1.815228,0.0
1,2.0,1854,0.370011,0.028212,-0.036861,2.950545,-0.347634,123.104500,70.798726,1.738795,0.0
2,2.5,1854,0.407228,0.029913,-0.029508,2.950545,-0.348655,130.902871,75.444141,1.735097,0.0


,symbol,setup_name,trades,win_rate,avg_return,median_return,profit_factor,stop_rate,edge_tier,eligible,watchlist_eligible,stop_loss,holding_period,cost_assumption
0,HDB,breakout_20d,12,0.833333,0.534940,0.555902,60.437820,0.166667,Valid Edge,True,True,-0.05,20,0.004
1,SZC,breakout_20d,10,0.800000,0.239352,0.216193,23.162268,0.200000,Valid Edge,True,True,-0.05,20,0.004
2,NAF,breakout_20d,13,0.615385,0.056990,0.059369,3.743964,0.384615,Valid Edge,True,True,-0.05,20,0.004
3,DIG,breakout_20d,13,0.538462,0.035099,0.022284,2.408283,0.461538,Valid Edge,True,True,-0.05,20,0.004
4,CTS,breakout_20d,18,0.388889,0.023402,-0.054000,1.709146,0.611111,Valid Edge,True,True,-0.05,20,0.004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
476,CHPG2516,breakout_20d,1,0.000000,-0.054000,-0.054000,0.000000,1.000000,Insufficient Data,False,False,-0.05,20,0.004
477,CHPG2505,breakout_20d,2,0.000000,-0.054000,-0.054000,0.000000,1.000000,Insufficient Data,False,False,-0.05,20,0.004
478,SFG,breakout_20d,2,0.000000,-0.054000,-0.054000,0.000000,1.000000,Insufficient Data,False,False,-0.05,20,0.004
479,SGN,breakout_20d,2,0.000000,-0.028257,-0.028257,0.000000,0.500000,Insufficient Data,False,False,-0.05,20,0.004


,volatility_regime,setup_name,trades,win_rate,avg_return,median_return,profit_factor,stop_rate,volatility_rule,stop_loss,holding_period
0,Normal Vol,breakout_20d,169,0.520710,0.102885,0.023095,5.143318,0.372781,Allowed,-0.05,20
1,Low Vol,breakout_20d,98,0.520408,0.086486,0.020126,4.873663,0.367347,Allowed,-0.05,20
2,Panic Vol,breakout_20d,39,0.538462,0.061039,0.012370,3.449103,0.461538,Avoid,-0.05,20
3,Stress Vol,breakout_20d,67,0.417910,0.066270,-0.043337,3.303835,0.477612,Allowed,-0.05,20
4,High Vol,breakout_20d,137,0.401460,0.054656,-0.054000,2.623882,0.518248,Allowed,-0.05,20
5,Unknown,breakout_20d,1546,0.326003,0.030580,-0.054000,1.910504,0.589263,Ignore,-0.05,20


In [7]:
# Cell: Save Breakout Outputs
save_breakout_research_outputs(results, out_dir="data/signals", prefix="breakout_20d")
print("Saved breakout research artifacts to data/signals")


Saved breakout research artifacts to data/signals
